In [1]:
!mkdir -p models
!cp best_model.pth models/best_model.pth
!git add models/best_model.pth
!git commit -m "Add best model checkpoint"
!git push origin main

cp: cannot stat 'best_model.pth': No such file or directory
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [2]:
!git clone https://github.com/lshek22/facial-expression-recognition.git
%cd facial-expression-recognition
!pip install wandb -q

Cloning into 'facial-expression-recognition'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 30 (delta 7), reused 18 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 67.86 KiB | 1.01 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/facial-expression-recognition


In [3]:
import os, json
from google.colab import userdata

os.makedirs('/root/.config/kaggle', exist_ok=True)
kaggle_creds = {
    "username": 'lukashekiladze',
    "key": 'b2f5f196e6fac2129c67a38910282180'
}
with open('/root/.config/kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)
os.system('chmod 600 /root/.config/kaggle/kaggle.json')

!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q challenges-in-representation-learning-facial-expression-recognition-challenge.zip
!ls

100% 285M/285M [00:01<00:00, 157MB/s]

challenges-in-representation-learning-facial-expression-recognition-challenge.zip
example_submission.csv
experiment_deep_cnn.ipynb
experiment_resnet18.ipynb
experiment_simple_cnn.ipynb
fer2013.tar.gz
icml_face_data.csv
README.md
test.csv
train.csv


In [4]:
import torch
import torch.nn as nn
import torchvision.models as models

class ResNet18FER(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super(ResNet18FER, self).__init__()
        self.resnet = models.resnet18(weights=None)
        self.resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(in_features, 7)
        )

    def forward(self, x):
        return self.resnet(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/best_model.pth best_model.pth

model = ResNet18FER(dropout_rate=0.5).to(device)
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()
print("Model loaded successfully")

Mounted at /content/drive
Model loaded successfully!


In [5]:
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader

test_df = pd.read_csv('test.csv')
print(f"Test samples: {len(test_df)}")

class FERTestDataset(Dataset):
    def __init__(self, df):
        self.images = df['pixels'].apply(
            lambda x: np.array(x.split(), dtype=np.float32).reshape(48, 48)
        ).values

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx] / 255.0
        img = torch.tensor(img).unsqueeze(0)
        return img

test_dataset = FERTestDataset(test_df)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

Test samples: 7178


In [6]:
all_preds = []

with torch.no_grad():
    for imgs in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)

print(f"Total predictions: {len(all_preds)}")

Total predictions: 7178


In [7]:
sample_sub = pd.read_csv('example_submission.csv')
print("Expected format:")
print(sample_sub.head())
print(sample_sub.columns.tolist())

Expected format:
   3
0  4
1  0
2  4
3  3
4  3
['3']


In [8]:
submission = pd.DataFrame({
    'id': range(len(all_preds)),
    'emotion': all_preds
})

submission.to_csv('submission.csv', index=False)
print(submission.head(10))
print(f"Shape: {submission.shape}")

from google.colab import files
files.download('submission.csv')

   id  emotion
0   0        0
1   1        6
2   2        0
3   3        6
4   4        3
5   5        3
6   6        2
7   7        4
8   8        4
9   9        5
Shape: (7178, 2)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
!ls models/

ls: cannot access 'models/': No such file or directory
